In [1]:
# Jupyter helper: run a multi-line shell script from a Python string

import os
import subprocess
from typing import Optional, Dict

def run_block(
    script: str,
    *,
    shell: str = "/bin/bash",
    strict: bool = True,     # set -euo pipefail
    xtrace: bool = False,    # set -x (echo commands)
    cwd: Optional[str] = None,
    env: Optional[Dict[str, str]] = None,
    raise_on_error: bool = True
) -> int:
    """
    Execute a multi-line shell script string in the given shell.

    Streams combined stdout/stderr to the notebook output.
    Returns the process return code (0 on success).
    """
    prelude = []
    if strict:
        prelude.append("set -euo pipefail")
    if xtrace:
        prelude.append("set -x")
    final_script = ("\n".join(prelude) + "\n" if prelude else "") + script

    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [shell, "-lc", final_script],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=cwd,
        env=merged_env,
        bufsize=1  # line-buffered
    )

    # Stream output line-by-line to notebook
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()
    if rc != 0 and raise_on_error:
        raise subprocess.CalledProcessError(rc, final_script)
    return rc


In [2]:
CMDS = r"""
terraform -chdir=infra/docker init
terraform -chdir=infra/docker apply -auto-approve

"""

run_block(CMDS, xtrace=True)  # set xtrace=False if you don't want each command echoed



+ terraform -chdir=infra/docker init


Initializing the backend...
Initializing modules...
Initializing provider plugins...
- Reusing previous version of kreuzwerker/docker from the dependency lock file
- Using previously-installed kreuzwerker/docker v3.6.2

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.
+ terraform -chdir=infra/docker apply -auto-approve
var.airflow_admin_password
  Initial Airflow admin password.



KeyboardInterrupt: 